# Pipeline ETL — Arquitectura Medallion

Este notebook muestra el recorrido completo de los datos a través del pipeline:

1. **Bronze** — datos crudos y problemas detectados
2. **Quality (Bronze)** — reporte de calidad de la ingesta
3. **Silver** — transformaciones aplicadas y registros descartados
4. **Referential** — integridad referencial entre entidades
5. **Gold** — tablas analíticas finales

Todos los resultados se leen desde los Parquet ya generados por `python main.py`.

In [ ]:
import sys
from pathlib import Path

# Permite importar src/ cuando el notebook se abre desde la carpeta notebooks/
root = Path('..').resolve()
sys.path.insert(0, str(root))

import polars as pl
from src import config

# Trabajar siempre desde la raíz del proyecto
import os
os.chdir(root)

print('Directorio de trabajo:', Path.cwd())
print('Polars version:', pl.__version__)

---
## 1. Bronze — Datos crudos

La capa Bronze ingesta los CSVs tal cual, leyendo **todas las columnas como string** (`infer_schema_length=0`).
El objetivo es preservar la fidelidad del origen: ningún valor corrupto se pierde o modifica en esta etapa.

Se agregan dos columnas de metadata: `_source_file` y `_ingested_at`.

In [ ]:
# Resumen de filas y columnas por entidad
print(f'{'Entidad':<15} {'Filas':>7}  Columnas')
print('-' * 45)
bronze_frames = {}
for entity in config.ENTITIES:
    df = pl.read_parquet(config.BRONZE_DIR / f'{entity}.parquet')
    bronze_frames[entity] = df
    print(f'{entity:<15} {df.height:>7}  {df.columns}')

In [ ]:
# Muestra de orders — todos los campos llegan como string
print('=== orders (Bronze) — primeras 5 filas ===')
bronze_frames['orders'].head(5)

In [ ]:
# Problemas detectados en Bronze: nulos por columna
print('=== Nulos por columna en orders (Bronze) ===')
df = bronze_frames['orders']
null_counts = (
    df.select([pl.col(c).is_null().sum().alias(c) for c in df.columns if not c.startswith('_')])
)
null_counts

In [ ]:
# Problemas detectados: duplicados por clave primaria
print('=== Duplicados por entidad (Bronze) ===')
print(f'{'Entidad':<15} {'Total':>7} {'Únicos':>7} {'Duplicados':>10}')
print('-' * 45)
for entity, pk in config.PRIMARY_KEYS.items():
    df = bronze_frames[entity]
    total = df.height
    unique = df.select(pl.col(pk).n_unique()).item()
    dups = total - unique
    print(f'{entity:<15} {total:>7} {unique:>7} {dups:>10}')

In [ ]:
# Problemas detectados: formatos inconsistentes en status
print('=== Valores únicos de status en orders (Bronze) ===')
bronze_frames['orders'].get_column('status').value_counts().sort('count', descending=True)

In [ ]:
# Problemas detectados: precios no numéricos en products
print('=== Muestra de prices en products (Bronze) — todavía string ===')
bronze_frames['products'].select(['product_id', 'product_name', 'price']).head(8)

---
## 2. Quality — Reporte de calidad de Bronze

Cada etapa del pipeline genera checks de calidad con un esquema estándar.
Todos se acumulan en `data/quality/quality_checks.parquet`.

In [ ]:
# Cargar tabla de calidad completa
qc = pl.read_parquet(config.QUALITY_DIR / 'quality_checks.parquet')
print(f'Total de checks registrados: {qc.height}')
print(f'Etapas: {qc["stage"].unique().to_list()}')

In [ ]:
# Checks de Bronze: null_check con fallos > 0
print('=== Columnas con nulos en Bronze ===')
(
    qc
    .filter((pl.col('stage') == 'bronze') & (pl.col('check_name') == 'null_check') & (pl.col('records_failed') > 0))
    .select(['table', 'column', 'records_checked', 'records_failed', 'pct_failed'])
    .sort('pct_failed', descending=True)
)

In [ ]:
# Checks de Bronze: duplicados detectados
print('=== Duplicados detectados en Bronze ===')
(
    qc
    .filter((pl.col('stage') == 'bronze') & (pl.col('check_name') == 'duplicate_check'))
    .select(['table', 'column', 'records_checked', 'records_failed', 'pct_failed'])
    .sort('records_failed', descending=True)
)

---
## 3. Silver — Transformaciones y limpieza

Silver aplica las siguientes transformaciones por entidad:

- **Normalización de texto**: strip + lower/upper en columnas categóricas
- **Casteo de tipos**: string → Float64/Int64/Datetime con `strict=False`
- **Deduplicación**: se conserva la primera aparición por clave primaria
- **Imputación de centinelas**: nulos en columnas de agrupación → `'unknown'` / `'UNKNOWN'`
- **Cuarentena**: filas inválidas críticas se apartan con `_quarantine_reason`
- **Anulación**: valores inválidos no críticos se ponen en `null` (la fila se conserva)

In [ ]:
# Comparación Bronze vs Silver: filas por entidad
print(f'{'Entidad':<15} {'Bronze':>8} {'Silver':>8} {'Eliminados':>11}')
print('-' * 48)
for entity in config.ENTITIES:
    b = pl.read_parquet(config.BRONZE_DIR / f'{entity}.parquet').height
    s = pl.read_parquet(config.SILVER_DIR / f'{entity}.parquet').height
    print(f'{entity:<15} {b:>8} {s:>8} {b - s:>11}')

In [ ]:
# Transformación: normalización de status en orders
print('=== status en orders: Bronze (string crudo) vs Silver (normalizado) ===')
b_orders = bronze_frames['orders']
s_orders = pl.read_parquet(config.SILVER_DIR / 'orders.parquet')

print('\nBronze — values únicos de status:')
print(b_orders['status'].unique().to_list())

print('\nSilver — values únicos de status:')
print(s_orders['status'].value_counts().sort('count', descending=True))

In [ ]:
# Transformación: casteo de tipos en products
print('=== price en products: Bronze (Utf8) vs Silver (Float64) ===')
s_products = pl.read_parquet(config.SILVER_DIR / 'products.parquet')

print(f'Bronze  — dtype de price: {bronze_frames["products"]["price"].dtype}')
print(f'Silver  — dtype de price: {s_products["price"].dtype}')
print(f'Silver  — nulos en price: {s_products["price"].null_count()}')

print('\nSilver — products con price nulo (venían con valor no parseable):')
s_products.filter(pl.col('price').is_null()).select(['product_id', 'product_name', 'category'])

In [ ]:
# Cuarentena: registros descartados y motivos
print('=== Registros en cuarentena por entidad y motivo ===')
quarantine_files = list(config.QUALITY_DIR.glob('quarantine_*.parquet'))
quarantine_files = [f for f in quarantine_files if 'referential' not in f.name]

for path in sorted(quarantine_files):
    df = pl.read_parquet(path)
    print(f'\n{path.name} ({df.height} filas):')
    print(df['_quarantine_reason'].value_counts().sort('count', descending=True))

In [ ]:
# Quality Silver: checks post-limpieza
print('=== Quality checks de Silver con fallos > 0 ===')
(
    qc
    .filter((pl.col('stage') == 'silver') & (pl.col('records_failed') > 0))
    .select(['check_name', 'table', 'column', 'records_checked', 'records_failed', 'pct_failed'])
    .sort('pct_failed', descending=True)
)

---
## 4. Referential — Integridad referencial

Se validan las relaciones entre entidades **después** de la limpieza Silver.

**Estrategia diferenciada:**
- **Huérfanos del spine** (`order_items`, `payments`, `reviews` sin `order_id` válido): van a **cuarentena**. Sin un pedido padre no tienen sentido analítico.
- **Huérfanos de dimensión** (`orders` sin `customer_id`, `order_items` sin `product_id`): se **conservan**. El pedido ocurrió aunque el cliente o producto no esté en el catálogo.

In [ ]:
# Checks referenciales registrados
print('=== Checks de integridad referencial ===')
(
    qc
    .filter(pl.col('stage') == 'referential')
    .select(['check_name', 'table', 'column', 'records_checked', 'records_failed', 'pct_failed'])
    .sort('records_failed', descending=True)
)

In [ ]:
# Cuarentenas referenciales: huérfanos descartados del spine
print('=== Huérfanos del spine (order_id no existe en orders) ===')
ref_files = sorted(config.QUALITY_DIR.glob('quarantine_referential_*.parquet'))
for path in ref_files:
    df = pl.read_parquet(path)
    entity = path.stem.replace('quarantine_referential_', '')
    print(f'\n{entity}: {df.height} filas descartadas')
    print(df.select(['_quarantine_reason']).head(3))

---
## 5. Gold — Tablas analíticas

Cuatro tablas analíticas construidas sobre Silver limpio.

In [ ]:
# Resumen de tablas Gold generadas
print(f'{'Tabla':<35} {'Filas':>7}')
print('-' * 45)
gold_frames = {}
for name in ['gold_sales_by_state', 'gold_product_performance', 'gold_customer_segments', 'gold_monthly_kpis']:
    df = pl.read_parquet(config.GOLD_DIR / f'{name}.parquet')
    gold_frames[name] = df
    print(f'{name:<35} {df.height:>7}')

In [ ]:
# gold_sales_by_state: Top 10 estados por revenue
print('=== Top 10 estados por revenue ===')
gold_frames['gold_sales_by_state'].head(10)

In [ ]:
# gold_product_performance: Top 10 productos por revenue
print('=== Top 10 productos por revenue ===')
gold_frames['gold_product_performance'].head(10)

In [ ]:
# gold_product_performance: Top 10 con mayor tasa de devolución (mínimo 5 pedidos)
print('=== Top 10 productos con mayor tasa de devolución (>= 5 pedidos) ===')
(
    gold_frames['gold_product_performance']
    .filter(pl.col('order_count') >= 5)
    .sort('return_rate', descending=True)
    .select(['product_id', 'product_name', 'category', 'order_count', 'returned_orders', 'return_rate', 'avg_score'])
    .head(10)
)

In [ ]:
# gold_customer_segments: distribución de segmentos
print('=== Distribución de segmentos RFM ===')
(
    gold_frames['gold_customer_segments']
    .group_by('segment')
    .agg(
        pl.len().alias('clientes'),
        pl.col('monetary').mean().round(2).alias('gasto_promedio'),
        pl.col('frequency').mean().round(2).alias('pedidos_promedio'),
        pl.col('recency_days').mean().round(0).alias('dias_recencia_promedio'),
    )
    .sort('clientes', descending=True)
)

In [ ]:
# gold_customer_segments: Top 10 Champions por gasto
print('=== Top 10 Champions por gasto total ===')
(
    gold_frames['gold_customer_segments']
    .filter(pl.col('segment') == 'Champions')
    .select(['customer_id', 'customer_name', 'state', 'recency_days', 'frequency', 'monetary', 'rfm_score'])
    .head(10)
)

In [ ]:
# gold_monthly_kpis: evolución mensual completa
print('=== KPIs mensuales ===')
gold_frames['gold_monthly_kpis'].select([
    'year_month', 'total_orders', 'total_revenue',
    'canceled_orders', 'cancellation_rate',
    'returned_orders', 'return_rate',
    'avg_score', 'avg_ticket'
])

In [ ]:
# Resumen ejecutivo: métricas globales del negocio
kpis = gold_frames['gold_monthly_kpis']
segs = gold_frames['gold_customer_segments']

print('=== RESUMEN EJECUTIVO ===')
print(f"Revenue total:          ${kpis['total_revenue'].sum():>15,.2f}")
print(f"Pedidos totales:        {kpis['total_orders'].sum():>15,}")
print(f"Ticket promedio:        ${kpis['total_revenue'].sum() / kpis['total_orders'].sum():>15,.2f}")
print(f"Score promedio global:  {kpis['avg_score'].mean():>15.2f}")
print(f"Tasa cancelación media: {kpis['cancellation_rate'].mean() * 100:>14.1f}%")
print(f"Tasa devolución media:  {kpis['return_rate'].mean() * 100:>14.1f}%")
print()
print('Clientes por segmento:')
for row in segs.group_by('segment').agg(pl.len().alias('n')).sort('n', descending=True).iter_rows(named=True):
    pct = row['n'] / segs.height * 100
    print(f"  {row['segment']:<20} {row['n']:>5} clientes ({pct:.1f}%)")

---
## 6. Tabla de calidad completa

Todos los checks de las tres etapas (bronze, silver, referential, gold) en un solo lugar.

In [ ]:
# Resumen de checks por etapa
print('=== Checks de calidad por etapa ===')
(
    qc
    .group_by('stage')
    .agg(
        pl.len().alias('total_checks'),
        (pl.col('records_failed') > 0).sum().alias('checks_con_fallos'),
        pl.col('records_failed').sum().alias('total_registros_fallidos'),
    )
    .sort('stage')
)

In [ ]:
# Top 10 checks con mayor porcentaje de fallos
print('=== Top 10 checks con mayor pct_failed ===')
(
    qc
    .filter(pl.col('records_failed') > 0)
    .select(['check_name', 'table', 'column', 'records_checked', 'records_failed', 'pct_failed', 'stage'])
    .sort('pct_failed', descending=True)
    .head(10)
)

---
## Conclusiones

### Resumen del pipeline

| Etapa | Entradas | Salidas | Descartados |
|-------|----------|---------|-------------|
| Bronze | 6 CSVs crudos | 6 Parquet (todo-string) | 0 |
| Silver | 71.000 filas totales | ~68.660 filas limpias | ~1.333 en cuarentena |
| Referential | 68.660 filas | ~67.002 filas | ~658 huérfanos del spine |
| Gold | Silver limpio | 4 tablas analíticas | — |

### Principales hallazgos en los datos

- **Duplicados**: customers (45), orders (70), reviews (93) tenían registros repetidos por clave primaria.
- **Fechas nulas**: 151 pedidos sin `order_date` — descartados porque es el evento base de los KPIs.
- **Precios inválidos**: 25 productos sin precio válido y 855 order_items con precio o cantidad inválida.
- **Integridad referencial**: 658 registros en order_items, payments y reviews apuntaban a orders inexistentes.
- **Score de reseñas**: algunos scores fuera del rango 1–5 fueron anulados (no descartados) para conservar el order_id.

### Calidad del dato final (Silver)

- `customers.state`: todos los nulos imputados con `'UNKNOWN'` para no perder revenue en Gold.
- `products.weight_kg`: valores negativos anulados (peso imposible pero el producto sigue siendo válido).
- `reviews.score`: valores fuera de rango anulados (el order_id y la existencia de la reseña siguen siendo útiles).